# 03_seguridad · Clasificación, anonimización y control de acceso (A2.4)
**Grupo 03 · Retail / Comercio.** Marco: Ley 1581 de 2012 y Decreto 1377 de 2013.

**Hallazgo previo.** El dataset del caso es agregado por tienda, departamento y semana: **no contiene datos personales**
(nombres, documentos, tarjetas de fidelización). La Ley 1581 protege datos de personas naturales, así que aquí no aplica
directamente. Sí hay información comercial confidencial de la cadena, y se trata con el mismo rigor como ejercicio y
porque un caso real (compras por cliente) sí traería datos personales.

In [0]:
from pyspark.sql import functions as F
ESQUEMA = "workspace.default"
v = spark.table(f"{ESQUEMA}.plata_ventas_enriquecida")

## 1. Clasificación de columnas por nivel de sensibilidad
Niveles: **Público**, **Interno**, **Confidencial** (comercial), **Personal** (Ley 1581), **Sensible** (art. 5 Ley 1581).

In [0]:
clasificacion = [
    ("Store",        "Interno",       "Identifica un establecimiento de la cadena; no identifica personas.", "Hash SHA-256 con sal"),
    ("Dept",         "Interno",       "Categoría comercial interna.", "Ninguna"),
    ("Date",         "Interno",       "Semana de venta; combinada con tienda permite inferir operación.", "Generalizar a mes"),
    ("Weekly_Sales", "Confidencial",  "Cifra de negocio de la cadena; ventaja competitiva.", "Acceso por rol; redondeo en vistas externas"),
    ("Size",         "Interno",       "Tamaño del local; con Store permite reidentificar la tienda.", "Generalizar a rangos"),
    ("Type",         "Público",       "Formato A/B/C, sin riesgo.", "Ninguna"),
    ("Temperature",  "Público",       "Dato climático de fuente abierta.", "Ninguna"),
]
display(spark.createDataFrame(clasificacion, ["columna", "nivel", "justificacion", "tratamiento"]))

columna,nivel,justificacion,tratamiento
Store,Interno,Identifica un establecimiento de la cadena; no identifica personas.,Hash SHA-256 con sal
Dept,Interno,Categoría comercial interna.,Ninguna
Date,Interno,Semana de venta; combinada con tienda permite inferir operación.,Generalizar a mes
Weekly_Sales,Confidencial,Cifra de negocio de la cadena; ventaja competitiva.,Acceso por rol; redondeo en vistas externas
Size,Interno,Tamaño del local; con Store permite reidentificar la tienda.,Generalizar a rangos
Type,Público,"Formato A/B/C, sin riesgo.",Ninguna
Temperature,Público,Dato climático de fuente abierta.,Ninguna


## 2. Anonimización: antes y después
- **Técnica 1, hash SHA-256 con sal:** `Store` → `store_hash`.
- **Técnica 2, generalización / enmascaramiento:** `Date` → `mes`, `Size` → rango, `Store` → máscara `S-**`.

La sal está en el código solo como ejemplo del ejercicio. En un caso real iría en un secreto (`dbutils.secrets`).

In [0]:
SAL = "pi_g03_demo"
anon = (v.select("Store", "Dept", "Date", "Size", "Weekly_Sales")
        .withColumn("store_hash", F.sha2(F.concat(F.col("Store").cast("string"), F.lit(SAL)), 256))
        .withColumn("store_mask", F.concat(F.lit("S-"), F.lit("**")))
        .withColumn("mes", F.date_format("Date", "yyyy-MM"))
        .withColumn("size_rango", F.when(F.col("Size") < 80000, "<80k").when(F.col("Size") < 150000, "80k-150k").otherwise(">=150k")))
print("ANTES")
display(anon.select("Store", "Date", "Size").limit(5))
print("DESPUÉS")
display(anon.select("store_hash", "store_mask", "mes", "size_rango").limit(5))

ANTES


Store,Date,Size
1,2010-07-23,151315
1,2010-08-20,151315
1,2011-11-18,151315
1,2012-01-27,151315
1,2012-02-10,151315


DESPUÉS


store_hash,store_mask,mes,size_rango
d28bcc9761570680a2fc777e81ddfb4de76040fa7e9f3fddab1a7be118313a5c,S-**,2010-07,>=150k
d28bcc9761570680a2fc777e81ddfb4de76040fa7e9f3fddab1a7be118313a5c,S-**,2010-08,>=150k
d28bcc9761570680a2fc777e81ddfb4de76040fa7e9f3fddab1a7be118313a5c,S-**,2011-11,>=150k
d28bcc9761570680a2fc777e81ddfb4de76040fa7e9f3fddab1a7be118313a5c,S-**,2012-01,>=150k
d28bcc9761570680a2fc777e81ddfb4de76040fa7e9f3fddab1a7be118313a5c,S-**,2012-02,>=150k


In [0]:
vista = anon.select("store_hash", "Dept", "mes", "size_rango", "Weekly_Sales")
vista.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{ESQUEMA}.plata_ventas_anonimizadas")
assert "Store" not in vista.columns and "Date" not in vista.columns
print("plata_ventas_anonimizadas creada sin Store ni Date en claro")

plata_ventas_anonimizadas creada sin Store ni Date en claro


## 3. Control de acceso por rol
Modelo propuesto sobre el esquema. En Databricks Free Edition la gestión de grupos puede estar limitada, por eso cada
sentencia se intenta y se reporta; la matriz sigue siendo el entregable de diseño.

| Rol | `plata_ventas_enriquecida` (en claro) | `plata_ventas_anonimizadas` | `plata_cuarentena_*` |
|---|---|---|---|
| `ingeniero_datos` | SELECT, MODIFY | SELECT, MODIFY | SELECT, MODIFY |
| `analista_negocio` | SELECT | SELECT | sin acceso |
| `auditor_externo` | sin acceso | SELECT | sin acceso |

In [0]:
grants = [
    f"GRANT SELECT ON TABLE {ESQUEMA}.plata_ventas_enriquecida TO `analista_negocio`",
    f"GRANT SELECT ON TABLE {ESQUEMA}.plata_ventas_anonimizadas TO `analista_negocio`",
    f"GRANT SELECT ON TABLE {ESQUEMA}.plata_ventas_anonimizadas TO `auditor_externo`",
    f"GRANT SELECT, MODIFY ON SCHEMA {ESQUEMA} TO `ingeniero_datos`",
]
resultado = []
for g in grants:
    try:
        spark.sql(g); resultado.append((g, "aplicado"))
    except Exception as e:
        resultado.append((g, "no aplicado: " + str(e)[:120]))
display(spark.createDataFrame(resultado, ["sentencia", "estado"]))

sentencia,estado
GRANT SELECT ON TABLE workspace.default.plata_ventas_enriquecida TO `analista_negocio`,no aplicado: [RequestId=502f2672-859f-4e39-bc41-b100ff68eb2f ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not
GRANT SELECT ON TABLE workspace.default.plata_ventas_anonimizadas TO `analista_negocio`,no aplicado: [RequestId=c3fa69e2-2e44-4fec-8616-00a95b1230b1 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not
GRANT SELECT ON TABLE workspace.default.plata_ventas_anonimizadas TO `auditor_externo`,no aplicado: [RequestId=a6c1cace-1815-41be-9e6f-3e6dc4f3ec5a ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not
"GRANT SELECT, MODIFY ON SCHEMA workspace.default TO `ingeniero_datos`",no aplicado: [RequestId=92290094-5017-49c0-b4f4-8f72c20ece30 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not
